# Room Redesign — SDXL + ControlNet

**Input:** 1 ảnh phòng + 3 lựa chọn: **loại phòng**, **phong cách**, **giữ layout hay không**

**Output:** ảnh phòng render lại với nội thất đúng loại phòng bạn chọn, theo đúng phong cách

**Nguyên tắc:** *loại phòng luôn thắng*. Đồ đạc trong ảnh gốc bị xoá hết, thay bằng đồ đúng loại
phòng bạn chọn. Ảnh gốc chỉ còn dùng để giữ kiến trúc — tường, cửa sổ, tỉ lệ phòng.

| `keep_layout` | Giữ gì |
|---|---|
| **True** | Cửa sổ, đường chân tường, góc tường, chiều cao trần đứng đúng chỗ. Dùng khi ảnh gốc là phòng thật của khách. |
| **False** | Chỉ giữ đại khái hình dạng phòng, model được đổi cả vị trí cửa sổ / mảng tường. Ảnh thường đẹp hơn nhưng không còn là phòng đó nữa. |

**Kỹ thuật:** SDXL + checkpoint nội thất (RealVisXL / Juggernaut XL) + ControlNet-MLSD
(chỉ trích đường thẳng kiến trúc, nên đồ đạc cũ không dính vào control map).

> **Bắt buộc:** Runtime > Change runtime type > **GPU** (T4 là đủ).
> Thời gian: ~60-120 giây/ảnh trên T4 free.


## 1. Cài đặt thư viện

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors peft
!pip install -q controlnet_aux opencv-python-headless


## 2. Import & load model

Lần đầu chạy sẽ tải ~8-10 GB (SDXL + 2 ControlNet), mất 3-6 phút.

- `BASE_MODEL`: checkpoint SDXL. RealVisXL V5 và Juggernaut XL v9 đều mạnh về nội thất/kiến trúc; `stabilityai/stable-diffusion-xl-base-1.0` là bản gốc (an toàn nhất nhưng ảnh "AI" hơn).
- `use_small_controlnet`: bản ControlNet rút gọn của diffusers, nhẹ VRAM hơn hẳn — nên bật trên T4 free. Tắt nếu chạy trên A100/L4 để có độ bám sát cao hơn.


In [ ]:
#@title Load SDXL + ControlNet { display-mode: "form" }
BASE_MODEL = "SG161222/RealVisXL_V5.0"  #@param ["SG161222/RealVisXL_V5.0", "RunDiffusion/Juggernaut-XL-v9", "stabilityai/stable-diffusion-xl-base-1.0"]
use_small_controlnet = True  #@param {type:"boolean"}
#@markdown `load_depth_controlnet`: ControlNet-Depth giữ khối 3D của đồ đạc cũ. Cách làm hiện tại luôn
#@markdown xoá đồ cũ nên **không cần depth** -> để tắt cho nhẹ VRAM. Chỉ bật nếu bạn tự sửa code để giữ đồ.
load_depth_controlnet = False  #@param {type:"boolean"}

import gc
import numpy as np
import torch
from PIL import Image
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
    UniPCMultistepScheduler,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
if device == "cpu":
    print("CẢNH BÁO: không thấy GPU. Runtime > Change runtime type > GPU, rồi chạy lại.")

if use_small_controlnet:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0-small"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0-small"
else:
    CANNY_ID = "diffusers/controlnet-canny-sdxl-1.0"
    DEPTH_ID = "diffusers/controlnet-depth-sdxl-1.0"

# ---- depth estimator (chỉ cần khi xử lý phòng đã có đồ) ----
depth_estimator = None
if load_depth_controlnet:
    from controlnet_aux import MidasDetector
    print("Đang tải depth estimator (MiDaS)...")
    depth_estimator = MidasDetector.from_pretrained("lllyasviel/Annotators")

# ---- MLSD line detector: chi bat duong THANG DAI (tuong, cua so, tran)
#      -> dung cho che do doi cong nang phong, vi do dac cu bi loai khoi control map
from controlnet_aux import MLSDdetector
print("Đang tải MLSD line detector...")
mlsd_detector = MLSDdetector.from_pretrained("lllyasviel/Annotators")

# ---- ControlNet ----
print(f"Đang tải ControlNet-Canny: {CANNY_ID}")
controlnet_canny = ControlNetModel.from_pretrained(CANNY_ID, torch_dtype=dtype, variant=None)

controlnet_depth = None
if load_depth_controlnet:
    print(f"Đang tải ControlNet-Depth: {DEPTH_ID}")
    controlnet_depth = ControlNetModel.from_pretrained(DEPTH_ID, torch_dtype=dtype, variant=None)

# ---- VAE: bản fp16-fix, tránh ảnh ra đen/NaN khi chạy fp16 trên T4 ----
print("Đang tải VAE fp16-fix...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)

# ---- Pipeline: nạp danh sách 2 controlnet, lúc gọi sẽ chọn dùng 1 hay 2 ----
controlnets = [controlnet_canny] + ([controlnet_depth] if controlnet_depth is not None else [])

print(f"Đang tải SDXL: {BASE_MODEL}")
def _load_pipe(variant):
    return StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_MODEL,
        controlnet=controlnets,
        vae=vae,
        torch_dtype=dtype,
        use_safetensors=True,
        variant=variant,
    )

# Nhieu checkpoint community khong up ban fp16 rieng -> thu fp16 truoc, khong co thi lay ban full
try:
    pipe = _load_pipe("fp16" if dtype == torch.float16 else None)
except Exception as e:
    print(f"  Khong co variant fp16 ({type(e).__name__}), tai ban day du...")
    pipe = _load_pipe(None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Tiết kiệm VRAM trên T4 16GB: offload từng module sang CPU khi không dùng
pipe.enable_model_cpu_offload()

# VAE slicing/tiling: diffusers moi bo 2 method nay o cap pipeline, chuyen xuong pipe.vae
for _name in ("enable_slicing", "enable_tiling"):
    if hasattr(pipe.vae, _name):
        getattr(pipe.vae, _name)()
    elif hasattr(pipe, f"enable_vae_{_name.split('_')[1]}"):
        getattr(pipe, f"enable_vae_{_name.split('_')[1]}")()

gc.collect()
torch.cuda.empty_cache() if device == "cuda" else None

CONTROL_ORDER = ["canny"] + (["depth"] if controlnet_depth is not None else [])
print(f"\nModel sẵn sàng. ControlNet đang nạp: {CONTROL_ORDER}")


## 3. Room type & Style
- `ROOM_TYPES`: loại phòng -> mô tả tiếng Anh + nội thất đặc trưng (giúp model đặt đúng đồ vào đúng phòng).
- `STYLE_PROMPTS`: phong cách -> mô tả vật liệu / màu / ánh sáng.
- Prompt cuối = style + room + nội thất + quality tag. Thêm loại phòng / style mới thì thêm 1 dòng vào dict, và thêm vào danh sách `#@param` ở bước 5.


In [ ]:
import re

# ---------- 3.1 Loại phòng ----------
# key = nhãn hiển thị, value = (mô tả phòng tiếng Anh, nội thất đặc trưng)
ROOM_TYPES = {
    "Phòng khách":       ("living room",            "large sofa, coffee table, armchair, area rug, floor lamp, framed wall art, curtains"),
    "Phòng ngủ":         ("bedroom",                "double bed with headboard, bedding, nightstands, bedside lamps, wardrobe, area rug, curtains"),
    "Phòng ăn":          ("dining room",            "dining table, dining chairs, pendant light above table, sideboard, centerpiece"),
    "Phòng tắm":         ("bathroom",               "bathtub, toilet, vanity with mirror, walk-in shower, wall tiles, towels"),
    "Bếp":               ("kitchen",                "kitchen cabinets, stone countertop, kitchen island, bar stools, tile backsplash, range hood"),
    "Phòng chơi game":   ("gaming room",            "gaming desk, gaming chair, dual monitors, led strip lighting, wall shelves, bean bag"),
    "Nhà hàng":          ("restaurant interior",    "multiple dining tables and chairs, bar counter, decorative pendant lights, banquette seating"),
    "Văn phòng tại gia": ("home office",            "desk, ergonomic chair, bookshelf, task lamp, potted plants, framed art"),
    "Quán cà phê":       ("coffee shop interior",   "cafe counter, espresso machine, small round tables, bentwood chairs, menu board, pendant lights"),
    "Văn phòng":         ("corporate office",       "workstations, office desks, task chairs, meeting table, acoustic panels, carpet tiles"),
}

# ---------- 3.2 Phong cách ----------
STYLE_PROMPTS = {
    # QUY TẮC: chỉ mô tả VẬT LIỆU / MÀU / ÁNH SÁNG / KHÔNG KHÍ.
    # Không được gọi tên món đồ (sofa, giường, thảm, rèm, tranh...) - đó là việc của
    # ROOM_TYPES. Style gọi tên đồ phòng khách sẽ nhét thảm len vào bếp, vào phòng tắm.
    "Peaceful":         "peaceful serene style, warm taupe and greige walls, matte natural finishes, "
                        "woven linen and wool textures, soft diffused daylight, muted olive and terracotta accents, dark walnut wood",
    "Farmhouse":        "modern farmhouse style, shiplap walls, reclaimed wood beams, warm neutral tones, "
                        "aged white paint, vintage patina, cozy atmosphere",
    "Clean Bright":     "clean bright style, crisp white walls, bright even daylight, light oak wood, "
                        "uncluttered, fresh and airy atmosphere",
    "Contemporary":     "contemporary style, sleek forms, mixed textures, muted palette with one bold accent color, "
                        "matte black details, designer lighting",
    "Fresh Airy":       "fresh airy style, pale palette, abundant daylight, light sheer fabrics, "
                        "indoor greenery, breezy open atmosphere",
    "Eclectic":         "eclectic style, bold color mix, patterned textiles, gallery wall, "
                        "mix of vintage and modern, playful energetic atmosphere",
    "Elegant":          "elegant style, soft neutral palette, silk and velvet textures, subtle gold details, "
                        "refined symmetry, polished surfaces",
    "Minimalist":       "minimalist style, clean lines, neutral color palette, hidden storage, "
                        "calm uncluttered surfaces, very few objects",
    "Minimal Tranquil": "minimal tranquil style, warm off-white tones, soft diffused light, natural linen, "
                        "zen calm, very sparse decoration",
    "Cartoon":          "cartoon illustration style, flat bold colors, thick outlines, playful stylized shapes, "
                        "cel shaded, 2d render",
    "Scandinavian":     "scandinavian style, light wood floor, white walls, cozy woven textiles, "
                        "hygge atmosphere, soft natural light",
    "Simple Calm":      "simple calm style, soft beige and grey palette, low profile forms, "
                        "uncluttered, gentle even lighting",
    "Bright Soothing":  "bright soothing style, pastel palette, soft rounded forms, warm sunlight, "
                        "comfortable relaxing atmosphere",
    "Cyberpunk":        "cyberpunk style, neon pink and cyan lighting, dark glossy surfaces, holographic panels, "
                        "futuristic tech details, moody atmosphere",
    "Rustic":           "rustic style, exposed wooden beams, stone wall, rough sawn timber, woven natural textiles, "
                        "earthy tones, aged patina",
    "Compact Calm":     "compact calm style, smart small space layout, multifunctional built-ins, "
                        "light palette, tidy and efficient",
    "Classic Graceful": "classic graceful style, wall moulding, cream and soft blue palette, "
                        "restrained ornament, crystal light fixtures, high ceiling",
    "Traditional":      "traditional style, dark stained wood, patterned textiles, symmetrical layout, "
                        "warm lamp light, framed wall decor",
    "Industrial":       "industrial style, exposed brick wall, black metal fixtures, concrete floor, "
                        "aged leather texture, edison bulb lighting",
    "Mid-Century":      "mid century modern style, teak wood, tapered legs, mustard and olive accents, "
                        "geometric patterns, 1960s design",
    "Japandi":          "japandi style, japanese and scandinavian fusion, natural wood, paper screen and woven straw texture, "
                        "soft neutral tones, minimal decoration",
    "Luxury Classic":   "luxury classic style, marble surfaces, gold accents, crystal light fixtures, "
                        "high ceiling, opulent polished finish",
}

# ---------- 3.3 Đồ đạc "sai phòng" ----------
# Cách làm: luôn ưu tiên loại phòng đã chọn, đồ trong ảnh gốc bị xoá hết. Nhưng dấu vết
# đồ cũ vẫn còn trong ảnh, nên đẩy các món dễ bị model giữ lại nhất vào negative prompt.
# Danh sách này được lọc theo phòng đích: render phòng ngủ thì "bed" tự bị bỏ khỏi negative.
WRONG_FURNITURE = "sofa, couch, bed, dining table, bathtub, toilet, kitchen cabinets, office desk, bar counter"

# Chặn TÊN loại phòng khác, không chỉ đồ đạc. SDXL có prior rất mạnh về "living room":
# cấm sofa/couch thôi vẫn ra phòng khách không sofa. Phải cấm thẳng "living room".
# Bỏ "interior"/"room" khi so trùng để "restaurant interior" không bị coi là trùng
# với "coffee shop interior" chỉ vì cùng chữ "interior".
ROOM_NAME_STOPWORDS = {"interior", "room"}

# ---------- 3.4 Quality tag & negative prompt ----------
QUALITY_TAGS = ("photorealistic, professional interior photography, architectural digest, "
                "soft natural lighting, detailed material and surface texture, sharp focus, high resolution")
# style vẽ minh hoạ thì không cần photorealistic -> tag riêng
NON_PHOTO_STYLES = {"Cartoon"}
NON_PHOTO_TAGS = "high quality illustration, clean vector render, detailed"

# Phải nói rõ "đã bày đồ", nếu không model hay trả lại đúng cái phòng trống trong ảnh gốc
FURNISH_TAGS = "fully furnished, professionally staged, correct fixtures and furnishings for this room type"

NEGATIVE_PROMPT = (
    "blurry, low quality, jpeg artifacts, distorted, deformed furniture, watermark, text, logo, "
    "unrealistic proportions, warped walls, crooked lines, extra doors, extra windows, people, "
    "duplicate objects, cluttered mess, floating furniture, oversaturated"
)
NEGATIVE_EMPTY = "empty room, unfurnished, bare floor, no furniture, vacant"


# Từ đồng nghĩa: phòng khách cần "sofa" thì negative cũng phải bỏ "couch",
# nếu không prompt đòi sofa mà negative cấm couch -> model rối, sofa ra méo.
SYNONYMS = {
    "sofa": {"couch"},        "couch": {"sofa"},
    "bed": {"mattress"},      "mattress": {"bed"},
    "desk": {"workstation", "workstations"},
    "counter": {"countertop"}, "countertop": {"counter"},
}


def _words(text: str) -> set:
    words = set(re.findall(r"[a-z]+", text.lower()))
    return words | {syn for w in words for syn in SYNONYMS.get(w, ())}


def wrong_furniture_for(room_type: str) -> str:
    """Bỏ những món mà phòng ĐÍCH cũng cần, tránh prompt đòi mà negative cấm.

    Render phòng ngủ -> bỏ "bed" khỏi negative (đích cần "double bed").
    Render quán cà phê -> bỏ "bar counter" (đích cần "cafe counter").
    """
    room_en, furniture = ROOM_TYPES[room_type]
    target_words = _words(room_en) | _words(furniture)
    kept = [item.strip() for item in WRONG_FURNITURE.split(",")
            if item.strip() and not (_words(item) & target_words)]
    return ", ".join(kept)


def wrong_rooms_for(room_type: str) -> str:
    """Tên các loại phòng KHÁC, để đẩy vào negative prompt.

    Đây là thứ dập prior "living room" của SDXL - render phòng tắm mà không cấm
    "living room" thì ảnh ra cứ lãng đãng kiểu phòng khách ốp gạch.
    """
    room_en, furniture = ROOM_TYPES[room_type]
    target_words = (_words(room_en) | _words(furniture)) - ROOM_NAME_STOPWORDS
    others = []
    for name, (other_en, _) in ROOM_TYPES.items():
        if name == room_type:
            continue
        short = other_en.replace(" interior", "")
        if _words(short) - ROOM_NAME_STOPWORDS & target_words:
            continue
        others.append(short)
    return ", ".join(others)


def build_prompt(room_type: str, style: str) -> tuple:
    """Trả về (prompt, prompt_2).

    SDXL có 2 text encoder, mỗi cái chỉ nhận 77 token. Nhồi hết vào 1 chỗ thì
    diffusers cắt phần cuối -> mất luôn quality tag. Nên tách:
      prompt   = loại phòng + nội thất + style   (nội dung chính)
      prompt_2 = loại phòng + tag bày đồ + tag chất lượng

    LOẠI PHÒNG PHẢI ĐỨNG ĐẦU. CLIP đánh trọng số token đầu cao hơn hẳn: nếu để
    ~40 token style lên trước thì "bathroom interior" rơi xuống vị trí thứ 45 và
    bị style lấn, ảnh ra phòng nào cũng lãng đãng kiểu phòng khách.
    """
    assert room_type in ROOM_TYPES, f"Room type không hợp lệ. Chọn 1 trong: {list(ROOM_TYPES)}"
    assert style in STYLE_PROMPTS, f"Style không hợp lệ. Chọn 1 trong: {list(STYLE_PROMPTS)}"
    room_en, furniture = ROOM_TYPES[room_type]
    tags = NON_PHOTO_TAGS if style in NON_PHOTO_STYLES else QUALITY_TAGS
    prompt = f"{room_en} interior, {furniture}, {STYLE_PROMPTS[style]}"
    prompt_2 = f"{room_en}, {FURNISH_TAGS}, {tags}"
    return prompt, prompt_2


def build_negative(room_type: str) -> tuple:
    """Trả về (negative_prompt, negative_prompt_2)."""
    parts = [NEGATIVE_EMPTY, wrong_furniture_for(room_type)]
    rooms = wrong_rooms_for(room_type)
    if rooms:
        parts.append(rooms)
    return NEGATIVE_PROMPT, ", ".join(p for p in parts if p)


def count_tokens(text: str):
    """Đếm token bằng chính tokenizer của pipeline. None nếu chưa load model."""
    _pipe = globals().get("pipe", None)
    return None if _pipe is None else len(_pipe.tokenizer(text).input_ids)


def check_prompt(room_type: str, style: str, verbose: bool = True):
    """In prompt + số token, cảnh báo nếu vượt 77 (SDXL sẽ cắt phần vượt)."""
    prompt, prompt_2 = build_prompt(room_type, style)
    negative, negative_2 = build_negative(room_type)
    for label, text in (("prompt", prompt), ("prompt_2", prompt_2),
                        ("negative", negative), ("negative_2", negative_2)):
        n = count_tokens(text)
        flag = "  <-- VƯỢT 77 TOKEN, SDXL sẽ cắt phần cuối" if (n or 0) > 77 else ""
        if verbose:
            print(f"[{label}] {'?' if n is None else n} token{flag}\n  {text}\n")
    return prompt, prompt_2


# Từ chỉ món đồ gắn với 1 loại phòng cụ thể. Style chứa những từ này sẽ nhét đồ
# phòng khách vào bếp / phòng tắm, và đụng luôn negative prompt "đồ sai phòng".
STYLE_BANNED_WORDS = {
    "sofa", "couch", "upholstery", "bed", "mattress", "headboard", "rug", "curtains",
    "tatami", "fireplace", "bathtub", "toilet", "desk", "artwork", "triptych",
    "dining", "cabinets", "countertop", "island", "wardrobe", "nightstand",
}


def audit_styles() -> dict:
    """Kiểm tra style nào còn gọi tên món đồ. Trả về {style: [từ vi phạm]}."""
    return {name: sorted(_words(text) & STYLE_BANNED_WORDS)
            for name, text in STYLE_PROMPTS.items() if _words(text) & STYLE_BANNED_WORDS}


_violations = audit_styles()
if _violations:
    print("CẢNH BÁO - style đang gọi tên món đồ, sẽ nhét đồ sai vào phòng:")
    for name, words in _violations.items():
        print(f"  {name}: {words}")
else:
    print("Style đều chỉ mô tả vật liệu/màu/ánh sáng, không gọi tên món đồ. OK.")

print(f"{len(ROOM_TYPES)} loại phòng x {len(STYLE_PROMPTS)} style = {len(ROOM_TYPES) * len(STYLE_PROMPTS)} tổ hợp")
print("\nVí dụ - render thành Phòng tắm theo style Peaceful:\n")
check_prompt("Phòng tắm", "Peaceful")


## 4. Hàm xử lý chính

Chỉ còn **một cách xử lý**, điều khiển bằng đúng 1 công tắc `keep_layout`:

| | Control map | Weight | Kết quả |
|---|---|---|---|
| `keep_layout=True` | MLSD | 0.60 | Cửa sổ, chân tường, góc tường giữ đúng chỗ. Đồ đạc cũ vẫn bị xoá. |
| `keep_layout=False` | MLSD | 0.25 | Chỉ giữ đại khái khối phòng, model tự do đổi mảng tường / vị trí cửa sổ. |

**Vì sao dùng MLSD chứ không phải Canny hay Depth:** Canny bắt *mọi* đường biên nên giữ luôn
đường viền cái sofa cũ — model sẽ nhồi đồ mới vào đúng hình sofa đó, ra đồ méo. Depth thì giữ
nguyên khối 3D của đồ cũ, còn tệ hơn. MLSD chỉ bắt **đường thẳng dài** = gần như chỉ còn kiến trúc,
nên xoá đồ cũ rồi bày lại từ đầu mới sạch.

*Hạn chế:* đồ nội thất cũng có vài cạnh thẳng (mặt bàn, lưng sofa). Nếu ảnh ra vẫn còn dấu vết đồ cũ
thì nâng `mlsd_threshold` ở cell nâng cao lên 0.2-0.3.

Ảnh luôn được resize **giữ đúng tỉ lệ gốc** về ~1 MP (SDXL train quanh mức này), không bóp về vuông.


In [ ]:
# Thông số mặc định. Sửa ở cell "Thông số nâng cao" bên dưới, hoặc cứ để nguyên.
ADVANCED = {
    "line_scale_keep": 0.60,   # weight MLSD khi keep_layout = True
    "line_scale_free": 0.25,   # weight MLSD khi keep_layout = False
    "mlsd_threshold": 0.10,    # cao hơn = ít đường hơn = xoá đồ cũ sạch hơn
    "guidance_scale": 6.0,
    "steps": 30,
    "seed": 42,
    "target_px": 1024 * 1024,
}


def fit_size(w: int, h: int, target_px: int) -> tuple:
    """Giữ tỉ lệ gốc, scale về ~target_px, làm tròn về bội số 8 (yêu cầu của SDXL)."""
    ar = w / h
    new_h = (target_px / ar) ** 0.5
    new_w = ar * new_h
    return (max(512, int(round(new_w / 8) * 8)), max(512, int(round(new_h / 8) * 8)))


def make_mlsd(image: Image.Image, thr_v: float = 0.1, thr_d: float = 0.1) -> Image.Image:
    """Chỉ đường THẲNG DÀI -> gần như chỉ còn kiến trúc, đồ đạc bị loại khỏi control map."""
    out = mlsd_detector(
        image, thr_v=thr_v, thr_d=thr_d,
        detect_resolution=512, image_resolution=max(image.size),
    )
    return out.convert("RGB").resize(image.size, Image.LANCZOS)


def redesign_room(input_image_path: str, room_type: str, style: str, keep_layout: bool = True, **over):
    """Render lại phòng theo room_type + style. Đồ đạc trong ảnh gốc bị xoá hết.

    keep_layout: True = khoá cửa sổ / đường tường theo ảnh gốc, False = cho phép đổi.
    **over: ghi đè bất kỳ khoá nào trong ADVANCED cho riêng lần gọi này.
    """
    cfg = {**ADVANCED, **over}
    line_scale = cfg.get("line_scale") or (
        cfg["line_scale_keep"] if keep_layout else cfg["line_scale_free"])

    src = Image.open(input_image_path).convert("RGB")
    size = fit_size(*src.size, target_px=cfg["target_px"])
    room_image = src.resize(size, Image.LANCZOS)

    line_image = make_mlsd(room_image, thr_v=cfg["mlsd_threshold"])

    # Pipeline được nạp với 1 hoặc 2 ControlNet -> phải truyền đủ ảnh cho từng cái.
    # Depth (nếu có nạp) luôn để weight 0 vì cách làm này không giữ đồ đạc cũ.
    control_images = [line_image]
    control_scales = [line_scale]
    if "depth" in CONTROL_ORDER:
        control_images.append(Image.new("RGB", size, (0, 0, 0)))
        control_scales.append(0.0)

    prompt, prompt_2 = build_prompt(room_type, style)
    negative, negative_2 = build_negative(room_type)

    result = pipe(
        prompt=prompt,
        prompt_2=prompt_2,
        negative_prompt=negative,
        negative_prompt_2=negative_2,
        image=control_images,
        controlnet_conditioning_scale=control_scales,
        num_inference_steps=cfg["steps"],
        guidance_scale=cfg["guidance_scale"],
        width=size[0],
        height=size[1],
        generator=torch.Generator(device="cpu").manual_seed(cfg["seed"]),
    ).images[0]

    return room_image, line_image, result


## 5. Thông số nâng cao — bỏ qua được

Cell này chỉ ghi đè `ADVANCED`. **Không chạy cũng được**, mặc định đã hợp lý.
Chỉ mở ra khi ảnh kết quả có vấn đề cụ thể:

| Triệu chứng | Sửa |
|---|---|
| Ảnh còn dấu vết đồ cũ (sofa mờ trong phòng ngủ) | `mlsd_threshold` lên **0.20-0.30** |
| Đổi sang phòng tắm / bếp / văn phòng mà ảnh vẫn "la lá phòng khách" | đặt `keep_layout = False`, và `mlsd_threshold` lên **0.25** — hình học phòng khách trong ảnh gốc đang lấn |
| Tường / cửa sổ bị méo, lệch | `line_scale_keep` lên **0.75-0.85** |
| Phòng vẫn ít đồ | `line_scale_keep` xuống **0.40-0.50** |
| Ảnh trông "AI", màu bệt, viền gắt | `guidance_scale` xuống **4.5-5.5** và `steps` lên **40** |
| `CUDA out of memory` | `resolution` sang **0.6 MP** |
| Muốn xem phương án bố trí đồ khác | đổi `seed` |


In [ ]:
#@title Thông số nâng cao (không cần sửa) { display-mode: "form" }

#@markdown **`mlsd_threshold`** — ngưỡng nhận đường thẳng. Cao hơn = giữ ít đường hơn = **xoá đồ cũ sạch hơn**, nhưng quá cao thì mất cả đường kiến trúc và phòng bắt đầu méo.
mlsd_threshold = 0.1  #@param {type:"slider", min:0.05, max:0.4, step:0.05}
#@markdown **`line_scale_keep`** — mức bám kiến trúc khi `keep_layout = True`. Cao = tường/cửa sổ đúng chỗ tuyệt đối nhưng model bị bó, ít đồ hơn.
line_scale_keep = 0.6  #@param {type:"slider", min:0.2, max:1.0, step:0.05}
#@markdown **`line_scale_free`** — mức bám khi `keep_layout = False`. Để thấp cho model tự do đổi layout.
line_scale_free = 0.25  #@param {type:"slider", min:0.0, max:0.6, step:0.05}
#@markdown **`guidance_scale` (CFG)** — mức tuân prompt. RealVisXL/Juggernaut thích **4-7**; đẩy lên 10-12 là ảnh cháy màu, trông "AI" ngay.
guidance_scale = 6.0  #@param {type:"slider", min:3.0, max:10.0, step:0.5}
#@markdown **`steps`** — số bước khử nhiễu. 25-30 là đủ, 40+ nét hơn chút nhưng lâu gấp rưỡi.
steps = 30  #@param {type:"integer"}
#@markdown **`seed`** — cùng seed + cùng thông số = ra đúng ảnh cũ. Giữ nguyên khi đang tinh chỉnh để so sánh công bằng, đổi khi muốn phương án bố trí khác.
seed = 42  #@param {type:"integer"}
#@markdown **`resolution`** — ảnh luôn giữ đúng tỉ lệ gốc, chỉ scale về mức pixel này. Ảnh 16:9 ra ~1368x768 (1 MP) hoặc ~1056x592 (0.6 MP).
resolution = "1.0 MP (chất lượng)"  #@param ["1.0 MP (chất lượng)", "0.6 MP (nhanh)"]

ADVANCED.update({
    "mlsd_threshold": mlsd_threshold,
    "line_scale_keep": line_scale_keep,
    "line_scale_free": line_scale_free,
    "guidance_scale": guidance_scale,
    "steps": steps,
    "seed": seed,
    "target_px": 1024 * 1024 if resolution.startswith("1.0") else int(0.6 * 1024 * 1024),
})
for k, v in ADVANCED.items():
    print(f"  {k} = {v}")


## 6. Upload ảnh & render

Đúng 3 lựa chọn. Bấm **Choose Files** để chọn ảnh phòng (jpg/png/webp), chọn nhiều ảnh được.


In [ ]:
#@title Upload ảnh và render { display-mode: "form" }

#@markdown **`room_type`** — phòng bạn muốn ra. Quyết định **đồ gì được đưa vào**: chọn Phòng ngủ thì ra giường / tủ / đèn ngủ, kể cả khi ảnh gốc là phòng khách có sofa. Đồ trong ảnh gốc **bị xoá hết**.
room_type = "Phòng khách"  #@param ["Phòng khách", "Phòng ngủ", "Phòng ăn", "Phòng tắm", "Bếp", "Phòng chơi game", "Nhà hàng", "Văn phòng tại gia", "Quán cà phê", "Văn phòng"]

#@markdown **`style`** — phong cách. Quyết định **màu sắc, vật liệu, ánh sáng** của đồ đạc đó, không đổi loại đồ.
style = "Peaceful"  #@param ["Peaceful", "Farmhouse", "Clean Bright", "Contemporary", "Fresh Airy", "Eclectic", "Elegant", "Minimalist", "Minimal Tranquil", "Cartoon", "Scandinavian", "Simple Calm", "Bright Soothing", "Cyberpunk", "Rustic", "Compact Calm", "Classic Graceful", "Traditional", "Industrial", "Mid-Century", "Japandi"]

#@markdown **`keep_layout`** — có giữ kiến trúc của ảnh gốc hay không:
#@markdown - **True** — cửa sổ, đường chân tường, góc tường, chiều cao trần **đứng đúng chỗ**. Dùng khi đây là phòng thật của khách và họ cần thấy đúng phòng mình.
#@markdown - **False** — chỉ giữ đại khái hình dạng phòng, model được đổi vị trí cửa sổ và mảng tường. Ảnh thường đẹp hơn nhưng không còn là phòng đó nữa.
#@markdown
#@markdown ⚠️ **Đổi sang loại phòng khác hẳn ảnh gốc thì nên để `False`.** Ảnh gốc là phòng khách rộng, cửa sổ lớn, sàn trống; ép giữ đúng hình học đó rồi đòi ra *phòng tắm* thì kết quả vẫn là một cái sảnh rộng có bồn tắm — vì phòng tắm thật vốn nhỏ và kín. Cùng loại phòng (phòng khách → phòng khách, chỉ đổi style) thì để `True`.
keep_layout = True  #@param {type:"boolean"}

import os, time
from google.colab import files

print("Chọn 1 hoặc nhiều ảnh phòng để upload...")
uploaded = files.upload()

VALID_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
image_paths = []
for name in uploaded.keys():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Bỏ qua (không phải ảnh): {name}")
        continue
    path = os.path.join("/content", name)
    with open(path, "wb") as f:
        f.write(uploaded[name])
    image_paths.append(path)

assert image_paths, "Chưa có ảnh nào được upload. Chạy lại cell và chọn file ảnh."

_scale = ADVANCED["line_scale_keep"] if keep_layout else ADVANCED["line_scale_free"]
print(f"\n{room_type} | {style} | keep_layout={keep_layout} (mlsd weight={_scale})\n")
check_prompt(room_type, style)

results = []   # [(tên file, original, control, output), ...]
for path in image_paths:
    t0 = time.time()
    print(f"Đang xử lý: {os.path.basename(path)} ...")
    original, control, output = redesign_room(path, room_type, style, keep_layout=keep_layout)
    results.append((os.path.basename(path), original, control, output))
    print(f"  xong sau {time.time() - t0:.0f}s | {output.size[0]}x{output.size[1]}")

input_path = image_paths[-1]
_, original, control, output = results[-1]
print(f"\nXong {len(results)} ảnh.")


## 7. Hiển thị kết quả: gốc / đường kiến trúc (MLSD) / kết quả


In [ ]:
from PIL import Image as PILImage
from IPython.display import display

def show_side_by_side(*images, row_height=384):
    """Ghép ngang, chuẩn hoá theo chiều cao để không méo ảnh."""
    imgs = []
    for im in images:
        im = im.convert("RGB")
        w = int(im.width * row_height / im.height)
        imgs.append(im.resize((w, row_height), PILImage.LANCZOS))
    total_width = sum(im.width for im in imgs)
    combined = PILImage.new("RGB", (total_width, row_height), (255, 255, 255))
    x = 0
    for im in imgs:
        combined.paste(im, (x, 0))
        x += im.width
    return combined

for name, orig, ctrl, out in results:
    stem = name.rsplit(".", 1)[0]
    out.save(f"/content/{stem}_result.jpg", quality=95)
    combined = show_side_by_side(orig, ctrl, out)
    combined.save(f"/content/{stem}_compare.jpg", quality=95)
    print(f"{name} -> /content/{stem}_result.jpg  (+ _compare.jpg)")
    display(combined)
    display(out)
